In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter


In [2]:
# --- 1. 超参数配置 ---
hparams = {
    'dataset': 'car',
    'hidden_channels': 32,
    'heads': 4,
    'num_hgt_layers': 2,
    'topk_memberships_per_object': 64,
    'learning_rate': 0.005,
    'weight_decay': 5e-4,
    'epochs': 150,
    'dropout': 0.5,
    'seed': 42
}

torch.manual_seed(hparams['seed'])
np.random.seed(hparams['seed'])


In [3]:
# --- 2. TensorBoard 设置 ---
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
log_dir_name = f"../runs/{hparams['dataset']}_bipartite_object_concept_transformer_{timestamp}"
writer = SummaryWriter(log_dir_name)
print(f"TensorBoard 日志将保存在: {log_dir_name}")


TensorBoard 日志将保存在: ../runs/car_bipartite_object_concept_transformer_20260623-162424


In [4]:
# --- 3. 数据读取工具函数 ---
def load_labels(base_path, dataset_name, expected_num_nodes):
    candidate_paths = [
        f"{base_path}{dataset_name}.data",
        f"{base_path}{dataset_name}.data.csv",
    ]

    for labels_path in candidate_paths:
        try:
            df = pd.read_csv(labels_path, header=None)
        except FileNotFoundError:
            continue

        df = df.dropna(axis=1, how='all')
        if len(df) != expected_num_nodes:
            continue
        return df.iloc[:, -1].values

    raise ValueError(f"无法读取与对象数量 {expected_num_nodes} 匹配的标签文件: {candidate_paths}")


def load_feature_matrix(path):
    values = np.loadtxt(path, delimiter=',')
    if values.ndim == 1:
        values = values.reshape(1, -1)
    return torch.tensor(values, dtype=torch.float)


def keep_topk_memberships_per_object(df, topk):
    if topk is None or topk <= 0:
        return df
    if len(df) == 0:
        return df

    # 边权越大表示概念越具体。每个对象只保留最具体的 topK 个所属概念，控制二部图规模。
    return (df.sort_values(['object_id', 'weight', 'concept_id'], ascending=[True, False, True])
              .groupby('object_id', group_keys=False)
              .head(topk)
              .reset_index(drop=True))


def load_bipartite_edges(path, object_count, concept_count, topk_per_object=None):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        gz_path = path + '.gz'
        df = pd.read_csv(gz_path, compression='gzip')
    required_columns = {'object_id', 'concept_id', 'weight'}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"边表必须包含列 {required_columns}: {path}")

    original_edge_count = len(df)
    df = keep_topk_memberships_per_object(df, topk_per_object)

    object_ids = torch.tensor(df['object_id'].to_numpy(), dtype=torch.long)
    concept_ids = torch.tensor(df['concept_id'].to_numpy(), dtype=torch.long)
    weights = torch.tensor(df['weight'].to_numpy(), dtype=torch.float).view(-1, 1)

    if object_ids.numel() > 0:
        if object_ids.min() < 0 or object_ids.max() >= object_count:
            raise ValueError(f"对象 id 超出范围: {path}")
        if concept_ids.min() < 0 or concept_ids.max() >= concept_count:
            raise ValueError(f"概念 id 超出范围: {path}")

    obj_to_concept = torch.stack([object_ids, concept_ids], dim=0)
    concept_to_obj = torch.stack([concept_ids, object_ids], dim=0)
    return obj_to_concept, concept_to_obj, weights, weights.clone(), original_edge_count, len(df)


In [5]:
# --- 4. 构建对象-概念二部图 HeteroData ---
def load_and_prepare_bipartite_data(dataset_name, topk_memberships_per_object=None):
    base_path = f'../data/{dataset_name}/'

    x_raw = load_feature_matrix(f"{base_path}{dataset_name}.data.cleaned.csv")
    num_objects = x_raw.shape[0]

    pos_concept_x = load_feature_matrix(f"{base_path}{dataset_name}_positive_object_concept_concept_features.csv")
    neg_concept_x = load_feature_matrix(f"{base_path}{dataset_name}_negative_object_concept_concept_features.csv")

    pos_obj_to_concept, pos_concept_to_obj, pos_edge_attr, pos_rev_edge_attr, pos_original_edges, pos_kept_edges = load_bipartite_edges(
        f"{base_path}{dataset_name}_positive_object_concept_edges.csv",
        num_objects,
        pos_concept_x.shape[0],
        topk_memberships_per_object
    )
    neg_obj_to_concept, neg_concept_to_obj, neg_edge_attr, neg_rev_edge_attr, neg_original_edges, neg_kept_edges = load_bipartite_edges(
        f"{base_path}{dataset_name}_negative_object_concept_edges.csv",
        num_objects,
        neg_concept_x.shape[0],
        topk_memberships_per_object
    )

    labels_numpy = load_labels(base_path, dataset_name, num_objects)
    encoder = LabelEncoder()
    y_numpy = encoder.fit_transform(labels_numpy)
    y = torch.tensor(y_numpy, dtype=torch.long)

    data = HeteroData()
    data['object_pos'].x = x_raw
    data['object_neg'].x = x_raw.clone()
    data['concept_pos'].x = pos_concept_x
    data['concept_neg'].x = neg_concept_x
    data['object_pos'].y = y
    data['object_neg'].y = y.clone()

    data['object_pos', 'belongs_to_pos', 'concept_pos'].edge_index = pos_obj_to_concept
    data['object_pos', 'belongs_to_pos', 'concept_pos'].edge_attr = pos_edge_attr
    data['concept_pos', 'contains_pos', 'object_pos'].edge_index = pos_concept_to_obj
    data['concept_pos', 'contains_pos', 'object_pos'].edge_attr = pos_rev_edge_attr

    data['object_neg', 'belongs_to_neg', 'concept_neg'].edge_index = neg_obj_to_concept
    data['object_neg', 'belongs_to_neg', 'concept_neg'].edge_attr = neg_edge_attr
    data['concept_neg', 'contains_neg', 'object_neg'].edge_index = neg_concept_to_obj
    data['concept_neg', 'contains_neg', 'object_neg'].edge_attr = neg_rev_edge_attr

    num_train = int(num_objects * 0.6)
    num_val = int(num_objects * 0.2)
    indices = torch.randperm(num_objects)
    train_mask = torch.zeros(num_objects, dtype=torch.bool); train_mask[indices[:num_train]] = True
    val_mask = torch.zeros(num_objects, dtype=torch.bool); val_mask[indices[num_train:num_train + num_val]] = True
    test_mask = torch.zeros(num_objects, dtype=torch.bool); test_mask[indices[num_train + num_val:]] = True
    data['object_pos'].train_mask = train_mask
    data['object_pos'].val_mask = val_mask
    data['object_pos'].test_mask = test_mask

    print(f"对象原始特征维度: {x_raw.shape[1]}")
    print(f"正概念节点数: {pos_concept_x.shape[0]}, 正概念特征维度: {pos_concept_x.shape[1]}, 正边数: {pos_kept_edges}/{pos_original_edges}")
    print(f"负概念节点数: {neg_concept_x.shape[0]}, 负概念特征维度: {neg_concept_x.shape[1]}, 负边数: {neg_kept_edges}/{neg_original_edges}")

    return data, len(np.unique(y_numpy))


In [6]:
# --- 5. 定义对象-概念二部图 Transformer 模型 ---
class BipartiteObjectConceptTransformer(nn.Module):
    def __init__(self, metadata, raw_in_channels, pos_concept_channels, neg_concept_channels,
                 hidden_channels, out_channels, heads=4, num_layers=2, dropout=0.5):
        super(BipartiteObjectConceptTransformer, self).__init__()
        self.dropout = dropout

        self.object_pos_encoder = nn.Linear(raw_in_channels, hidden_channels)
        self.object_neg_encoder = nn.Linear(raw_in_channels, hidden_channels)
        self.concept_pos_encoder = nn.Linear(pos_concept_channels, hidden_channels)
        self.concept_neg_encoder = nn.Linear(neg_concept_channels, hidden_channels)

        self.convs = nn.ModuleList([
            HGTConv(hidden_channels, hidden_channels, metadata, heads=heads)
            for _ in range(num_layers)
        ])

        self.fusion_layer = nn.Linear(hidden_channels * 2, out_channels)

    def forward(self, data):
        x_dict = {
            'object_pos': self.object_pos_encoder(data['object_pos'].x),
            'object_neg': self.object_neg_encoder(data['object_neg'].x),
            'concept_pos': self.concept_pos_encoder(data['concept_pos'].x),
            'concept_neg': self.concept_neg_encoder(data['concept_neg'].x),
        }

        for conv in self.convs:
            x_dict = conv(x_dict, data.edge_index_dict)
            x_dict = {node_type: F.dropout(F.relu(x), p=self.dropout, training=self.training)
                      for node_type, x in x_dict.items()}

        h_combined = torch.cat([x_dict['object_pos'], x_dict['object_neg']], dim=1)
        return self.fusion_layer(h_combined)


In [7]:
# --- 6. 实例化数据和模型 ---
data, num_classes = load_and_prepare_bipartite_data(
    hparams['dataset'],
    hparams['topk_memberships_per_object']
)

model = BipartiteObjectConceptTransformer(
    metadata=data.metadata(),
    raw_in_channels=data['object_pos'].x.shape[1],
    pos_concept_channels=data['concept_pos'].x.shape[1],
    neg_concept_channels=data['concept_neg'].x.shape[1],
    hidden_channels=hparams['hidden_channels'],
    out_channels=num_classes,
    heads=hparams['heads'],
    num_layers=hparams['num_hgt_layers'],
    dropout=hparams['dropout']
)

optimizer = torch.optim.Adam(model.parameters(), lr=hparams['learning_rate'], weight_decay=hparams['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()


对象原始特征维度: 25
正概念节点数: 12640, 正概念特征维度: 12, 正边数: 110592/173785
负概念节点数: 19473, 负概念特征维度: 12, 负边数: 110592/5907821


In [8]:
# --- 7. 训练与评估函数 ---
def train(epoch):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    mask = data['object_pos'].train_mask
    loss = criterion(out[mask], data['object_pos'].y[mask])
    loss.backward()
    optimizer.step()
    writer.add_scalar('Loss/train', loss.item(), epoch)
    return loss.item()


def evaluate(epoch):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)
        y = data['object_pos'].y
        train_mask = data['object_pos'].train_mask
        val_mask = data['object_pos'].val_mask
        test_mask = data['object_pos'].test_mask

        train_acc = (pred[train_mask] == y[train_mask]).sum().item() / train_mask.sum().item()
        val_acc = (pred[val_mask] == y[val_mask]).sum().item() / val_mask.sum().item()
        test_acc = (pred[test_mask] == y[test_mask]).sum().item() / test_mask.sum().item()

        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/validation', val_acc, epoch)
        writer.add_scalar('Accuracy/test', test_acc, epoch)
        return train_acc, val_acc, test_acc


In [9]:
# --- 8. 主训练循环 ---
print("\n--- 开始训练 (对象-概念二部图 Transformer + 概念节点 CPE) ---")
for epoch in range(1, hparams['epochs'] + 1):
    loss = train(epoch)
    train_acc, val_acc, test_acc = evaluate(epoch)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

final_train_acc, final_val_acc, final_test_acc = evaluate(hparams['epochs'])
print('--- 训练完成 ---')
print(f'最终测试集准确率: {final_test_acc:.4f}')

metrics = {'accuracy/final_train': final_train_acc, 'accuracy/final_validation': final_val_acc, 'accuracy/final_test': final_test_acc}
writer.add_hparams(hparams, metrics)
writer.close()



--- 开始训练 (对象-概念二部图 Transformer + 概念节点 CPE) ---


Epoch: 001, Loss: 1.3466, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


Epoch: 002, Loss: 1.3191, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


Epoch: 003, Loss: 1.2946, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


Epoch: 004, Loss: 1.2676, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


...

Epoch: 146, Loss: 0.0105, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


Epoch: 147, Loss: 0.0069, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


Epoch: 148, Loss: 0.0070, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


Epoch: 149, Loss: 0.0128, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


Epoch: 150, Loss: 0.0071, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000
--- 训练完成 ---
最终测试集准确率: 1.0000
